# Database-style Operations on Dataframes

### About the data

In this notebook, we will using daily weather data that was taken from the [National Centers for Environmental Information (NCEI) API](https://www.ncdc.noaa.gov/cdo-web/webservices/v2) . The data collection notebook contains
the process that was followed to collect the data.

    
Note: The NCEI is part of the National Oceanic and Atmospheric Administration (NOAA) and, as you can see from the URL for the API, this resource was created when the
NCEI was called the NCDC. Should the URL for this resource change in the future, you can search for the NCEI weather API to find the updated one.

### Background on the data

Data meanings:

- <b>PRCP </b>: precipitation in millimeters
- <b>SNOW </b>: snowfall in millimeters
- <b>SNWD </b>: snow depth in millimeters
- <b>TMAX </b>: maximum daily temperature in Celsius
- <b>TMIN </b>: minimum daily temperature in Celsius
- <b>TOBS </b> : temperature at time of observation in Celsius
- <b> WESF </b> : water equivalent of snow in millimeters

### Setup

In [1]:
import pandas as pd
weather = pd.read_csv('nyc_weather_2018.csv')
weather.head()

,date,datatype,station,attributes,value
0,2018-01-01T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,0800",0.0
1,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0015,",,N,1050",0.0
2,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,1050",0.0
3,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0017,",,N,0920",0.0
4,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,0920",0.0


### Querying DataFrames

The query() method is an easier way of filtering based on some criteria. For example, we can use it to find all entries where snow was recorded:

In [2]:
snowdata = weather.query('datatype == "SNOW" and value > 0')
snowdata.head()

,date,datatype,station,attributes,value
127,2018-01-01T00:00:00,SNOW,GHCND:US1NYWC0019,",,N,1700",25.0
816,2018-01-04T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,1600",229.0
819,2018-01-04T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,0830",10.0
823,2018-01-04T00:00:00,SNOW,GHCND:US1NJBG0018,",,N,0910",46.0
830,2018-01-04T00:00:00,SNOW,GHCND:US1NJES0018,",,N,0700",10.0


This is equivalent to quering the data/weather.db SQLite database for SELECT * FROM weather WHERE datatype == "SNOW" AND value > 0 :

In [4]:
import sqlite3

with sqlite3.connect('weather.db') as connection:
    snow_data_from_db = pd.read_sql(
        'SELECT * FROM weather WHERE datatype == "SNOW" AND value > 0',
        connection
    )
    
snowdata.reset_index().drop(columns='index').equals(snow_data_from_db)

True

Note this is also equivalent to creating Boolean masks:

In [5]:
weather[(weather.datatype == 'SNOW') & (weather.value > 0)].equals(snowdata)

True

### Merging DataFrames

We have data for many different stations each day; however, we don't know what the stations are just their IDs. We can join the data in the data/weather_stations.csv
file which contains information from the stations endpoint of the NCEI API. Consult the weather_data_collection.ipynb notebook to see how this was collected. It looks like this:

In [7]:
station_info = pd.read_csv('weather_stations.csv')
station_info.head()

,id,name,latitude,longitude,elevation
0,GHCND:US1CTFR0022,"STAMFORD 2.6 SSW, CT US",41.064100,-73.577000,36.6
1,GHCND:US1CTFR0039,"STAMFORD 4.2 S, CT US",41.037788,-73.568176,6.4
2,GHCND:US1NJBG0001,"BERGENFIELD 0.3 SW, NJ US",40.921298,-74.001983,20.1
3,GHCND:US1NJBG0002,"SADDLE BROOK TWP 0.6 E, NJ US",40.902694,-74.083358,16.8
4,GHCND:US1NJBG0003,"TENAFLY 1.3 W, NJ US",40.914670,-73.977500,21.6


As a reminder, the weather data looks like this:

In [8]:
weather.head()

,date,datatype,station,attributes,value
0,2018-01-01T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,0800",0.0
1,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0015,",,N,1050",0.0
2,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,1050",0.0
3,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0017,",,N,0920",0.0
4,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,0920",0.0


We can join our data by matching up the station_info.id column with the weather.station column. Before doing that though, let's see how many unique values we have:

In [9]:
station_info.id.describe()

count                   330
unique                  330
top       GHCND:US1CTFR0022
freq                      1
Name: id, dtype: object

While station_info has one row per station, the weather dataframe has many entries per station. Notice it also has fewer uniques:

In [10]:
weather.station.describe()

count                 12000
unique                  111
top       GHCND:USW00014732
freq                    888
Name: station, dtype: object

When working with joins, it is important to keep an eye on the row count. Some join types will lead to data loss:

In [11]:
station_info.shape[0], weather.shape[0]

(330, 12000)

In [12]:
def get_row_count(*dfs):
  return [df.shape[0] for df in dfs]
get_row_count(station_info, weather)

[330, 12000]

The map() function is more efficient than list comprehensions. We can couple this with getattr() to grab any attribute for multiple dataframes

In [13]:
def get_info(attr, *dfs):
  return list(map(lambda x: getattr(x, attr), dfs))
get_info('shape', station_info,weather)

[(330, 5), (12000, 5)]

By default merge() performs an inner join. We simply specify the columns to use for the join. The left dataframe is the one we call merge() on, and the right one is passed in as an argument:

In [14]:
inner_join = weather.merge(station_info,left_on='station', right_on='id')
inner_join.sample(5, random_state=0)

,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation
4954,2018-05-04T00:00:00,AWND,GHCND:USW00054787,",,W,",4.1,GHCND:USW00054787,"FARMINGDALE REPUBLIC AIRPORT, NY US",40.734430,-73.416370,22.8
11999,2018-12-04T00:00:00,WSF5,GHCND:USW00094789,",,W,",13.9,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.639150,-73.763900,2.7
11041,2018-12-01T00:00:00,PRCP,GHCND:US1NJMN0081,",,N,0700",0.0,GHCND:US1NJMN0081,"RED BANK 0.6 ENE, NJ US",40.351849,-74.057205,14.0
7532,2018-08-03T00:00:00,PRCP,GHCND:US1NYNS0046,",,N,0700",0.0,GHCND:US1NYNS0046,"MASSAPEQUA PARK 1.2 N, NY US",40.698077,-73.449893,10.7
11966,2018-12-04T00:00:00,PRCP,GHCND:USW00094741,",,W,",0.0,GHCND:USW00094741,"TETERBORO AIRPORT, NJ US",40.858980,-74.056160,0.8


We can remove the duplication of information in the station and id columns by renaming one of them before the merge and then simply using on :

In [15]:
weather.merge(station_info.rename(dict(id='station'),axis=1),on='station').sample(5, random_state=0)

,date,datatype,station,attributes,value,name,latitude,longitude,elevation
4954,2018-05-04T00:00:00,AWND,GHCND:USW00054787,",,W,",4.1,"FARMINGDALE REPUBLIC AIRPORT, NY US",40.734430,-73.416370,22.8
11999,2018-12-04T00:00:00,WSF5,GHCND:USW00094789,",,W,",13.9,"JFK INTERNATIONAL AIRPORT, NY US",40.639150,-73.763900,2.7
11041,2018-12-01T00:00:00,PRCP,GHCND:US1NJMN0081,",,N,0700",0.0,"RED BANK 0.6 ENE, NJ US",40.351849,-74.057205,14.0
7532,2018-08-03T00:00:00,PRCP,GHCND:US1NYNS0046,",,N,0700",0.0,"MASSAPEQUA PARK 1.2 N, NY US",40.698077,-73.449893,10.7
11966,2018-12-04T00:00:00,PRCP,GHCND:USW00094741,",,W,",0.0,"TETERBORO AIRPORT, NJ US",40.858980,-74.056160,0.8


We are losing stations that don't have weather observations associated with them, if we don't want to lose these rows, we perform a right or left join instead of the inner join:

In [16]:
left_join = station_info.merge(weather, left_on='id',right_on='station', how='left')
right_join = weather.merge(station_info, left_on='station',right_on='id', how='right')

right_join.tail()

,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation
12214,2018-12-04T00:00:00,TMIN,GHCND:USW00094789,",,W,2400",-1.0,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.7639,2.7
12215,2018-12-04T00:00:00,WDF2,GHCND:USW00094789,",,W,",340.0,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.7639,2.7
12216,2018-12-04T00:00:00,WDF5,GHCND:USW00094789,",,W,",310.0,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.7639,2.7
12217,2018-12-04T00:00:00,WSF2,GHCND:USW00094789,",,W,",11.2,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.7639,2.7
12218,2018-12-04T00:00:00,WSF5,GHCND:USW00094789,",,W,",13.9,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.7639,2.7


In [17]:
left_join.sort_index(axis=1).sort_values(['date','station']).reset_index().drop(columns='index').equals(
    right_join.sort_index(axis=1).sort_values(['date','station']).reset_index().drop(columns='index')
)
  

True

Note we have additional rows in the left and right joins because we kept all the stations that didn't have weather observations:

In [18]:
get_info('shape', inner_join, left_join, right_join)

[(12000, 10), (12219, 10), (12219, 10)]

If we query the station information for stations that have NY in their name, believing that to be all the stations that record weather data for NYC and perform an outer join, we can see where the mismatches occur:

In [21]:
outer_join = weather.merge(
    station_info[station_info.name.str.contains('NY')],
    left_on='station',right_on='id', how='outer', indicator=True
)
pd.concat([outer_join.sample(4, random_state=0),outer_join[outer_join.station.isna()].head(2)]) 

,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation,_merge
6032,2018-11-04T00:00:00,SNWD,GHCND:USC00281335,",,7,0700",0.0,NaN,NaN,NaN,NaN,NaN,left_only
8838,2018-11-01T00:00:00,RHAV,GHCND:USW00014734,",,W,",75.0,NaN,NaN,NaN,NaN,NaN,left_only
8165,2018-01-03T00:00:00,WDF2,GHCND:USW00014734,",,W,",20.0,NaN,NaN,NaN,NaN,NaN,left_only
3970,2018-01-03T00:00:00,PRCP,GHCND:US1NYNS0030,",,N,0800",0.0,GHCND:US1NYNS0030,"PLAINEDGE 0.4 WSW, NY US",40.721382,-73.484241,22.9,both
1032,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJHD0018,"KEARNY 1.7 NNW, NJ US",40.774342,-74.137109,25.6,right_only
2119,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJMS0036,"PARSIPPANY TROY HILLS TWP 2.1 E, NJ US",40.865600,-74.385100,64.3,right_only


These joins are equivalent to their SQL counterparts. Below is the inner join. Note that to use equals() you will have to do some manipulation of the dataframes to line them up:

In [22]:
import sqlite3 as sq3

with sq3.connect('weather.db') as connection:
  inner_join_from_db = pd.read_sql('SELECT * FROM weather JOIN stations ON weather.station == stations.id',connection)

inner_join_from_db.shape == inner_join.shape

True

Revisit the dirty data from the previous module.

In [23]:
dirty_data = pd.read_csv('dirty_data2.csv', index_col='date').drop_duplicates().drop(columns='SNWD')
dirty_data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'dirty_data2.csv'

We need to create two dataframes for the join. We will drop some unecessary columns as well for easier viewing:

In [31]:
valid_station = dirty_data.query('station != "?"').copy().drop(columns=['WESF','station'])
sta_with_wesf = dirty_data.query('station == "?"').copy().drop(columns=['station', 'TOBS', 'TMIN', 'TMAX'])

NameError: name 'dirty_data' is not defined

Our column for the join is the index in both dataframes, so we must specify left_index and right_index :

In [32]:
valid_station.merge(sta_with_wesf, left_index=True, right_index=True).query('WESF>0').head()

NameError: name 'valid_station' is not defined

The columns that existed in both dataframes, but didn't form part of the join got suffixes added to their names: _ x _ for columns from the left dataframe and _y_ for columns from the right dataframe. We can customize this with the suffixes argument:

In [33]:
valid_station.merge(sta_with_wesf, left_index=True, right_index=True, suffixes = ('','_?')).query('WESF>0').head()

NameError: name 'valid_station' is not defined

Since we are joining on the index, an easier way is to use the join() method instead of merge() . Note that the suffix parameter is now lsuffix for the left dataframe's suffix and rsuffix for the right one's:

In [34]:
valid_station.join(sta_with_wesf, rsuffix='_?').query('WESF >0').head()

NameError: name 'valid_station' is not defined

Joins can be very resource-intensive, so it's a good idea to figure out what type of join you need using set operations before trying the join itself. The pandas set operations are performed on the index, so whichever columns we will be joining on will need to be the index. Let's go back to the weather and station_info dataframes and set the station ID columns as the index:

In [24]:
weather.set_index('station', inplace=True)
station_info.set_index('id', inplace=True)

The intersection will tell us the stations that are present in both dataframes. The result will be the index when performing an inner join:

In [25]:
weather.index.intersection(station_info.index)

Index(['GHCND:US1CTFR0039', 'GHCND:US1NJBG0015', 'GHCND:US1NJBG0017',
       'GHCND:US1NJBG0018', 'GHCND:US1NJBG0023', 'GHCND:US1NJBG0030',
       'GHCND:US1NJBG0039', 'GHCND:US1NJBG0044', 'GHCND:US1NJES0018',
       'GHCND:US1NJES0024',
       ...
       'GHCND:US1NJBG0037', 'GHCND:US1NYNY0074', 'GHCND:US1NJES0031',
       'GHCND:USC00284987', 'GHCND:US1NJES0029', 'GHCND:US1NJMD0086',
       'GHCND:US1NJMD0088', 'GHCND:US1NJMN0081', 'GHCND:US1NJMS0097',
       'GHCND:US1NJES0033'],
      dtype='str', length=111)

In [26]:
weather.index.difference(station_info.index)

Index([], dtype='str')

We lose 114 stations from the station_info dataframe, however:

In [27]:
station_info.index.difference(weather.index)

Index(['GHCND:US1CTFR0022', 'GHCND:US1NJBG0001', 'GHCND:US1NJBG0002',
       'GHCND:US1NJBG0005', 'GHCND:US1NJBG0006', 'GHCND:US1NJBG0008',
       'GHCND:US1NJBG0011', 'GHCND:US1NJBG0012', 'GHCND:US1NJBG0013',
       'GHCND:US1NJBG0020',
       ...
       'GHCND:USC00308749', 'GHCND:USC00308946', 'GHCND:USC00309117',
       'GHCND:USC00309270', 'GHCND:USC00309400', 'GHCND:USC00309466',
       'GHCND:USC00309576', 'GHCND:USC00309580', 'GHCND:USW00014708',
       'GHCND:USW00014786'],
      dtype='str', length=219)

The symmetric difference will tell us what gets lost from both sides. It is the combination of the set difference in both directions: 216 + 114 = 330 which was station_info shape was

In [28]:
ny_in_name = station_info[station_info.name.str.contains('NY')]

ny_in_name.index.difference(weather.index).shape[0]\
+ weather.index.difference(ny_in_name.index).shape[0]\
== weather.index.symmetric_difference(ny_in_name.index).shape[0]

True

The union will show us everything that will be present after a full outer join. Note that since these are sets (which don't allow duplicates by definition), we must pass unique
entries for union:

In [29]:
weather.index.unique().union(station_info.index)

Index(['GHCND:US1CTFR0022', 'GHCND:US1CTFR0039', 'GHCND:US1NJBG0001',
       'GHCND:US1NJBG0002', 'GHCND:US1NJBG0003', 'GHCND:US1NJBG0005',
       'GHCND:US1NJBG0006', 'GHCND:US1NJBG0008', 'GHCND:US1NJBG0010',
       'GHCND:US1NJBG0011',
       ...
       'GHCND:USW00014708', 'GHCND:USW00014732', 'GHCND:USW00014734',
       'GHCND:USW00014786', 'GHCND:USW00054743', 'GHCND:USW00054787',
       'GHCND:USW00094728', 'GHCND:USW00094741', 'GHCND:USW00094745',
       'GHCND:USW00094789'],
      dtype='str', length=330)

Note that the symmetric difference is actually the union of the set differences:

In [30]:
ny_in_name = station_info[station_info.name.str.contains('NY')]
ny_in_name.index.difference(weather.index).union(weather.index.difference(ny_in_name.index)).equals(
weather.index.symmetric_difference(ny_in_name.index)
)

True